In [1]:
# =============================================
#                   IMPORT
# =============================================
from keras.optimizers                   import Adam
from keras.callbacks                    import EarlyStopping, ReduceLROnPlateau
from preProcessingData                  import PreProcessData
from model.componentModel                  import SystemModel
from model.componentModel                  import SubclassModelCheckpoint

from tqdm                               import tqdm
from datetime                           import datetime
from prettytable                        import PrettyTable
import plotly.graph_objects             as go
import tensorflow                       as tf
import numpy                            as np 
import pandas                           as pd
import matplotlib.pyplot                as plt
import os, warnings
import random
import json


# ====================================
# Set debug level and 
# ------------------------------------
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"  
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'  # or any {'0', '1', '2'}
warnings.filterwarnings('ignore') 

np.random.seed(1)
random.seed(1)
tf.random.set_seed(1)

# ====================================
# Config
# ------------------------------------
with open('utility/modelConfig.json', 'r') as configFile:
    config = json.load(configFile)
myConfig   = config["system"]

2024-11-26 18:40:54.668172: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-11-26 18:40:54.780203: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2024-11-26 18:40:54.780218: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2024-11-26 18:40:54.805862: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-26 18:40:55.509289: W tensorflow/stream_executor/platform/de

In [2]:
preData                                 = PreProcessData(config = myConfig)                    # Preprocessing data
cr1, cr2, cv2                           = preData.loadH5File(name = "withoutNonCriticalData1", keys = ["cr1", "cr2", "cv2"])                           # Load the data from critical file
withIDCr1, withIDCr2, withIDCv2, IdSys  = preData.attachIDPerEvent(df = [cr1,cr2,cv2], mainternanceAsCriticalSd = True)


# Add the time to the data
# -----------------------------
# Normalize
# normalizedCr1           = preData.normalizeBuilinFunction(withIDCr1, headline = preData.crusher1Value)   
# addedData               = preData.addition_var(withIDCr1) 

# handledValues           = preData.crusher1Value + preData.addEfficiencyCr1 + preData.addedManipulatedCr1
handledValues           = preData.conveyorValue + preData.crusher2ConveyorValue
# -----------------------------
# Normalize
normalizedCv2           = preData.normalizeBuilinFunction(withIDCv2, headline = handledValues)  
# -----------------------------
# Add the Cycle and RUL to 
# each component
withTimeDataCv2         = preData.addCycleAndRul(normalizedCv2)                        # Add the time to the data
# -----------------------------
# Extract before split
dataCv2                = withTimeDataCv2[handledValues]

Process:21522||2024-11-26 18:40:57,108-INFO [preProcessingData.py:loadH5File():503]:Loading H5 file ...
Process:21522||2024-11-26 18:40:58,530-INFO [preProcessingData.py:loadH5File():509]:Finish loading data from files ...
Process:21522||2024-11-26 18:41:15,870-INFO [preProcessingData.py:normalizeBuilinFunction():1358]:Normalizing the data ....


In [3]:
model       = SystemModel(myConfig) 
model.build(input_shape=(None, myConfig["seqLength"], myConfig["inFeatureDim"]))
model.summary()
model.load_weights(model_dir = "checkpoint/bestModel")


# (522340, 60, 32) (522340, 1) (522340, 60, 30) (522340, 1) (522340, 60, 2) (522340, 1) (522340, 1)

2024-11-26 18:41:16.445974: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-11-26 18:41:16.446170: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-11-26 18:41:16.446355: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2024-11-26 18:41:16.446422: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublas.so.11'; dlerror: libcublas.so.11: cannot open shared object file: No such file or directory
2024-11-26 18:41:16.446480: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not lo

The shape of sysHILayer (None, 4, 5)
The shape of sysHILayer (None, 4, 5)
Model: "system_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sensor_selection (SensorSel  multiple                 31        
 ection)                                                         
                                                                 
 encoder (Functional)        [(None, 1),               114806    
                              (None, 4, 5)]                      
                                                                 
 decoder (Functional)        (None, 64, 31)            114431    
                                                                 
 hi_system_and_rul (HiSystem  multiple                 6413      
 AndRul)                                                         
                                                                 
Total params: 235,692
Trainable params: 235,68

Process:21522||2024-11-26 18:41:17,744-INFO [componentModel.py:load_weights():808]:Loading the model at the direction: checkpoint/bestModel
Process:21522||2024-11-26 18:41:17,772-INFO [componentModel.py:load_weights():818]:Loaded the model at the direction:checkpoint/bestModel successfully!!!


In [4]:
results            = {}
results["cv2"]     = []
# results["cr2"]     = []
# results["cv2"]     = []
# results["sys"]     = []
results["rul"]     = []

probsAll        = []
binaryMaskAll   = []
trajectID       = withIDCv2["trajectoryID"].astype(int)
for idx in range(trajectID.max()):
    print(idx)
    count = (trajectID == (idx + 1)).sum() 
    if count <= preData.config['seqLength']:
        continue
    seqDataCr2                              = preData.getTensorData(dataList   = [dataCv2], 
                                                                    trajectId  = trajectID,
                                                                    seqLength  = preData.config['seqLength'], 
                                                                    overlap    = preData.config['seqLength']-1,
                                                                    getAtIdx   = idx + 1,
                                                                    typeInput  = "listOfPandas")
    
    zCr1, binaryMaskCr1, probsMaskCr1, predictedRul     = model.getHIandRUL(seqDataCr2[0])
    # print(predictedRul)
    results["cv2"].append(zCr1)
    results["rul"].append(predictedRul)

Process:21522||2024-11-26 18:41:17,783-INFO [preProcessingData.py:getTensorData():1132]:Spliting data into sequences now ...


0


Process:21522||2024-11-26 18:41:18,618-INFO [preProcessingData.py:getTensorData():1132]:Spliting data into sequences now ...


1


Process:21522||2024-11-26 18:41:20,390-INFO [preProcessingData.py:getTensorData():1132]:Spliting data into sequences now ...


2


... (100 outputs left out)

Process:21522||2024-11-26 18:42:07,992-INFO [preProcessingData.py:getTensorData():1132]:Spliting data into sequences now ...


79


Process:21522||2024-11-26 18:42:09,173-INFO [preProcessingData.py:getTensorData():1132]:Spliting data into sequences now ...


80


In [5]:
# results            = {}
# results["cr1"]     = []
# # results["cr2"]     = []
# # results["cv2"]     = []
# # results["sys"]     = []
# results["rul"]     = []

# probsAll        = []
# binaryMaskAll   = []
# trajectID       = withIDCr1["trajectoryID"].astype(int)
# for idx in range(trajectID.max()):
#     print(idx)
#     count = (trajectID == (idx + 1)).sum() 
#     if count <= preData.config['seqLength']:
#         continue
#     seqDataCr1                              = preData.getTensorData(dataList   =    [dataCr1], 
#                                                                     trajectId   = trajectID,
#                                                                     seqLength  = preData.config['seqLength'], 
#                                                                     overlap    = preData.config['seqLength']-1,
#                                                                     getAtIdx   = idx + 1,
#                                                                     typeInput  = "listOfPandas")
    
#     zCr1, binaryMaskCr1, probsMaskCr1, predictedRul     = model.getHIandRUL(seqDataCr1[0])
#     # print(predictedRul)
#     results["cr1"].append(zCr1)
#     results["rul"].append(predictedRul)

### MASK

##### Crusher1

In [6]:
ssTable                 = PrettyTable()
ssTable.field_names     = ["No","sensorName","probability", "binary"]
print(f"The number of selected sensor {sum(binaryMaskCr1)}")
for idx,idxSS in enumerate(handledValues):
    ssTable.add_row([idx,idxSS,probsMaskCr1[idx].numpy(),binaryMaskCr1[idx].numpy()])

print(ssTable)

The number of selected sensor 14.0
+----+-----------------------------------------------+---------------+--------+
| No |                   sensorName                  |  probability  | binary |
+----+-----------------------------------------------+---------------+--------+
| 0  |                  Corriente(A)                 |      1.0      |  1.0   |
| 1  |                Pesómetro(ton/h)               |      1.0      |  1.0   |
| 2  |                 Nivel_Tolva(%)                |  0.0020933782 |  0.0   |
| 3  |                  Potencia(kW)                 | 5.0560023e-09 |  0.0   |
| 4  |            Presión_Diferencial(psi)           |      1.0      |  1.0   |
| 5  |                Renimiento(ton)                |  7.911777e-10 |  0.0   |
| 6  |                 Velocidad(rpm)                |      1.0      |  1.0   |
| 7  |             Eje_Piñón_Vibración_1             | 5.1041598e-06 |  0.0   |
| 8  |             Eje_Piñón_Vibración_2             |   0.9999999   |  1.0   |
| 9  

##### Crusher 2

In [7]:
# ssTable                 = PrettyTable()
# ssTable.field_names     = ["No","sensorName","probability", "binary"]
# print(f"The number of selected sensor {sum(binaryMaskCr2)}")
# for idx,idxSS in enumerate(preData.crusher2Value):
#     ssTable.add_row([idx,idxSS,probsMaskCr2[idx].numpy(),binaryMaskCr2[idx].numpy()])

# print(ssTable)

##### ConveyorBelt 2

In [8]:
# ssTable                 = PrettyTable()
# ssTable.field_names     = ["No","sensorName","probability", "binary"]
# print(f"The number of selected sensor {sum(binaryMaskCv2)}")
# for idx,idxSS in enumerate(preData.conveyorValue):
#     ssTable.add_row([idx,idxSS,probsMaskCv2[idx].numpy(),binaryMaskCv2[idx].numpy()])

# print(ssTable)

### Health indicators

##### Crusher 1

In [9]:
print(len(results))

2


In [10]:
fig = {}
for idxResults in results.keys():
    fig[idxResults]     = go.Figure()
    current_time        = 0
    
    for idx, signal in enumerate(results[idxResults]):
        # print(signal)
        signal          = tf.squeeze(signal).numpy()
        # print(idx)
        # print(signal)
        time_vector     = np.arange(current_time, current_time + len(signal))
        current_time    += len(signal)
        
        # Add each signal as a separate trace
        fig[idxResults].add_trace(go.Scatter(
                                                x   = time_vector,
                                                y   = signal,
                                                mode= 'lines',
                                                name= f'Signal {idx + 1}'  # Label each signal for legend toggling
        ))
    # Customize layout for readability
    fig[idxResults].update_layout(
                                    title       = idxResults,
                                    xaxis_title = "Time (arbitrary units)",
                                    yaxis_title = "Signal Value",
                                    template    = "plotly_white",
                                    showlegend  = True  # Show legend to allow toggling each signal
    )
    # Display the plot
    fig[idxResults].show()


##### Crusher 2

In [11]:
# fig1             = go.Figure()
# current_time    = 0

# for idx, signal in enumerate(results["cr2"]):
#     signal          = tf.squeeze(signal).numpy()
#     time_vector     = np.arange(current_time, current_time + len(signal))
#     current_time    += len(signal)
    
#     # Add each signal as a separate trace
#     fig1.add_trace(go.Scatter(
#         x=time_vector,
#         y=signal,
#         mode='lines',
#         name=f'Signal {idx + 1}'  # Label each signal for legend toggling
#     ))

# # Customize layout for readability
# fig1.update_layout(
#     title="Continuous Time Series Plot with Toggle",
#     xaxis_title="Time (arbitrary units)",
#     yaxis_title="Signal Value",
#     template="plotly_white",
#     showlegend=True  # Show legend to allow toggling each signal
# )

# # Display the plot
# fig1.show()
